# XGBoost

[XGBoost](https://xgboost.readthedocs.io/) is a gradient-boosted decision-tree
library. Unlike TabICL it is a classic **train-from-scratch** model: it fits an
ensemble of trees to your data. This notebook:

1. installs `xgboost` into the `.venv-notebooks` kernel (via `uv`),
2. defines a small **regression** dataset inline,
3. trains an `XGBRegressor` and predicts on held-out rows.


## 1. Install XGBoost into this kernel

`.venv-notebooks` is a `uv`-managed venv (no `pip` inside it), so we install with
`uv pip install` pointed at **this kernel's** interpreter via `sys.executable`. We use
the CPU-only wheel (`xgboost-cpu`) to match MyTraL's stack; it imports as `xgboost`.

In [1]:
import sys

# install into the exact interpreter this notebook kernel is running
!uv pip install --python "{sys.executable}" xgboost-cpu scikit-learn

Using Python 3.12.10 environment at: /home/dvorka/p/mytral/git/mytral/.venv
Resolved 7 packages in 340ms                                         
Prepared 1 package in 238ms                                              
Installed 1 package in 2ms                                  
 + xgboost-cpu==3.3.0


## 2. Imports and version check

In [2]:
import numpy as np
import sklearn
import xgboost as xgb

print("xgboost     ", xgb.__version__)
print("scikit-learn", sklearn.__version__)

xgboost      3.3.0
scikit-learn 1.9.0


## 3. A regression dataset defined inline

A seeded synthetic regression table generated in the notebook (reproducible). A few
features drive a continuous target plus noise; we hold out some rows to predict.

In [3]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

X, y = make_regression(
    n_samples=500,
    n_features=6,
    n_informative=4,
    noise=10.0,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=8, random_state=42
)

print("train rows:", X_train.shape, "  predict rows:", X_test.shape)
print("first train row:", np.round(X_train[0], 3), "-> target", round(float(y_train[0]), 3))

train rows: (492, 6)   predict rows: (8, 6)
first train row: [ 1.317 -0.118  0.168  1.318 -1.007  1.14 ] -> target 239.489


In [10]:
X[:10]

array([[-0.42924442,  0.58855327, -2.21186191,  1.5334337 , -1.42395715,
        -0.26665233],
       [ 0.36139561,  1.53803657, -0.07201012,  1.0035329 ,  0.36163603,
        -0.64511975],
       [ 0.49097495,  0.73487779, -0.16712171,  0.28257995, -0.24869113,
         1.60734558],
       [-0.45001286,  1.25714922, -1.5171737 ,  0.75057917, -0.4161944 ,
        -1.13006934],
       [-3.24126734, -1.02438764,  0.44381943,  0.77463405, -0.92693047,
        -0.05952536],
       [-1.51787375, -0.3570292 ,  0.69553776,  0.84910211, -0.29396695,
        -0.07159925],
       [ 2.09972179, -0.2470257 , -0.47004215,  0.26587823, -0.43671974,
        -0.06613261],
       [-0.30920908, -0.75215641, -1.50472037,  0.76005596,  0.08243975,
        -1.4575515 ],
       [-2.70323229,  0.67787532,  0.97519763,  0.50109417,  0.18958162,
         1.00104609],
       [-1.24386324, -0.6929052 , -0.70199169, -0.66290092, -1.40260527,
         1.74957674]])

In [9]:
y[:10]

array([-100.19465365,   52.89825883,  172.90018137, -180.89144026,
         84.92791552,  134.50112163,  -28.77605292, -197.09746066,
        206.24793294,   14.81782748])

## 4. Train and predict

`fit()` trains the boosted-tree ensemble; `predict()` returns the regressed values.
We report per-row prediction vs. truth and the overall RMSE / R^2.

In [11]:
from sklearn.metrics import mean_squared_error, r2_score

model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

for i, (pred, true) in enumerate(zip(y_pred, y_test)):
    print(f"row {i}: pred={pred:8.2f}  true={true:8.2f}  err={pred - true:7.2f}")

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"\nRMSE: {rmse:.3f}   R^2: {r2_score(y_test, y_pred):.3f}")

row 0: pred=  -14.77  true=    4.97  err= -19.75
row 1: pred=  -95.96  true=  -57.72  err= -38.24
row 2: pred=  227.64  true=  175.85  err=  51.80
row 3: pred=  156.26  true=  118.65  err=  37.61
row 4: pred=  271.14  true=  267.66  err=   3.48
row 5: pred=  145.82  true=  162.24  err= -16.42
row 6: pred= -157.71  true= -187.61  err=  29.89
row 7: pred=  228.42  true=  206.89  err=  21.53

RMSE: 30.798   R^2: 0.954
